# Potions: a single-run example (hydrology + reactive transport)

This notebook runs **one** forward simulation of the East River showcase
model — hydrology (four water-storage zones) coupled to reach-scale reactive
transport (DOC + aqueous speciation) — and plots the results. No ensemble,
no K-lane ladder: just a plain single parameter set, run exactly like the
parity/calibration scripts do, then visually inspected.

**What you need to know:**
- The public entry point is `potions.run_hydro_rt_batched`. With a `(1, n_params)`
  input it is a plain single run — the batched API simply broadcasts to one lane.
- Config assembly reuses the showcase loaders (`east_river_forward`, raw data
  readers); those paths are appendable from this notebook via `sys.path`.
- Compute is CPU, `torch.float64` — the bit-exact scalar-parity configuration.

Run the cells top to bottom.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from pandas import DatetimeIndex

import matplotlib.pyplot as plt

# --- make the showcase loaders importable ---
# (potions is pip-installed editable; the config helpers live in scripts/)
SCRIPTS = str(Path.cwd() / "scripts")
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)

import east_river_forward as fwd  # noqa: E402
from data.raw.east_river import hbv_data, load_all_doc_data  # noqa: E402
from utils.parameter_ranges import load_input_data  # noqa: E402

import potions as pt  # noqa: E402

print(f"potions at {pt.__file__}")
print(f"torch {torch.__version__}  matplotlib {plt.matplotlib.__version__}")

ModuleNotFoundError: No module named 'east_river_forward'

## 1. Build the window config

Same construction as `scripts/parity_check_batched.py`: load the input data
for the showcase, grab the `er_phf0` DOC observations, and assemble a
length-consistent config over a short window. Here we use the same 2021-09
window the oracle runs use.

In [ ]:
WINDOW: DatetimeIndex = DatetimeIndex(
    pd.date_range("2021-09-01", periods=30, freq="D")  # 30-day window
)

# Fold the prebuilt showcase data into a window-consistent config.
inputs = load_input_data(
    pt.HbvLateralModel, test_wys=[2017, 2020], val_wy=2018, site="EastRiver"
)
site_data = hbv_data.load_site_data("EastRiver")
doc_df = (
    load_all_doc_data()
    .loc[lambda d: d.station == "er_phf0", "npoc"]
    .rename("DOC")
    .to_frame()
    / 1000.0
    / 12.01  # ug/L -> mmol/L (MW_C)
)
doc_df = doc_df.groupby(doc_df.index).mean()

cfg = fwd.build_window_config(inputs.cal_config, site_data, doc_df, WINDOW)

# Step count is the window length (all daily series align to it).
n_steps: int = len(WINDOW)
forc = cfg["forc"]
print(f"window: {WINDOW[0]} .. {WINDOW[-1]}  ({n_steps} days)")
print(f"zones: {list(cfg['zone_configs'].keys())}")
print(f"model: {cfg['model_type'].__name__}")

## 2. A single parameter set

Parameters are a flat `(n_params,)` vector spanning hydro + RT zone + initial
state columns (85 columns for `HbvLateralModel`). We draw one vector inside
the inner 60% of each calibrated bound — a deterministic, in-bounds point.

In [ ]:
rng = np.random.default_rng(0)
bounds = inputs.bounds
params = np.asarray(
    [
        float(rng.uniform(lo + 0.2 * (hi - lo), lo + 0.8 * (hi - lo)))
        for (lo, hi) in bounds.values()
    ],
    dtype=float,
)

num_hydro = pt.HbvLateralModel.get_num_hydro_parameters()
num_rt_zone = pt.HbvLateralModel.get_num_rt_parameters(cfg["zone_configs"])
print(f"n_params = {params.size}  ({num_hydro} hydro + {num_rt_zone} RT + init)")
assert params.size == num_hydro + num_rt_zone + (params.size - num_hydro - num_rt_zone)

## 3. Run the model (hydrology + reactive transport)

`run_hydro_rt_batched` with a single row `(1, n_params)` is a single-run
forward — no ensemble. We stay on CPU with `float64` for exact scalar parity.

In [ ]:
device = torch.device("cpu")
dtype = torch.float64

result = pt.run_hydro_rt_batched(
    params.reshape(1, -1),
    cfg,
    device,
    dtype=dtype,
    init_aq_conc=1e-3,
)

print("overflow_mask:", result.overflow_mask.tolist())
print(
    "q_sim shape:",
    tuple(result.q_sim.shape),
    "   doc_riv shape:",
    tuple(result.doc_riv.shape),
)

## 4. Inspect the outputs

Per-lane (here: the single lane) results are `q_sim` (streamflow, mm/day) and
`doc_riv` (flow-weighted river DOC, mmol/L), plus the same objective dicts the
scalar calibration path produces.

In [ ]:
lane = 0
q_sim = result.q_sim[lane].detach().cpu().numpy()
doc_riv = result.doc_riv[lane].detach().cpu().numpy()

print("--- hydrology objectives ---")
for k, v in result.hydro_objectives[lane].items():
    print(
        f"   {k:>16s}: {float(v[0]) if hasattr(v, '__len__') and not isinstance(v, str) else float(v):.5f}"
    )

print("\n--- RT DOC objectives ---")
for k, v in result.rt_doc_objectives[lane].items():
    print(
        f"   {k:>16s}: {float(v[0]) if hasattr(v, '__len__') and not isinstance(v, str) else float(v):.5f}"
    )

## 5. Plot the simulation

Streamflow series with the measured `q_meas` overlay, and the river DOC series
with the measured concentration overlay (both from the config we built).

In [ ]:
q_meas = np.asarray(cfg["q_meas"], dtype=float)
conc = cfg["conc_df"]["DOC"] if "DOC" in cfg["conc_df"] else None

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(WINDOW, q_sim, "-o", ms=3, label="simulated q")
ax1.plot(WINDOW, q_meas, "--", lw=1.2, label="measured q")
ax1.set_ylabel("streamflow [mm/d]")
ax1.legend()
ax1.set_title("single-run East River forward (K=1)")

ax2.plot(WINDOW, doc_riv, "-s", ms=3, label="simulated DOC_riv")
if conc is not None:
    ax2.plot(WINDOW, np.asarray(conc, dtype=float), "--", lw=1.2, label="measured DOC")
ax2.set_ylabel("river DOC [mmol/L]")
ax2.set_xlabel("date")
ax2.legend()

plt.tight_layout()
plt.show()

## Recap

- `potions.run_hydro_rt_batched((1, n_params), cfg, device, dtype)` = one
  hydrology + reactive-transport simulation.
- Results: `.q_sim` / `.doc_riv` time series, `.hydro_objectives` /
  `.rt_doc_objectives` per lane, `.overflow_mask` per lane.
- For exact scalar parity stay on CPU `float64`; the batched API solves any
  `K >= 1` the same way (a single run is just `K=1`).

This mirrors the proven paths in `scripts/parity_check_batched.py` and
`scripts/profile_batched.py` — a valid baseline to compare ensemble/GPU runs
against.